# **Bài 1: Khôi phục Masked Token (Masked Language Modeling)**

In [9]:
from transformers import pipeline
# 1. Tải pipeline "fill-mask"
# Pipeline này sẽ tự động tải một mô hình mặc định phù hợp (thường là một biến thể của BERT)
mask_filler = pipeline("fill-mask")

# 2. Câu đầu vào với token [MASK]
input_sentence = "Hanoi is the <mask> of Vietnam."

# 3. Thực hiện dự đoán
# top_k=5 yêu cầu mô hình trả về 5 dự đoán hàng đầu
predictions = mask_filler(input_sentence, top_k=5)

# 4. In kết quả
print(f"Câu gốc: {input_sentence}")
for pred in predictions:
		print(f"Dự đoán: '{pred['token_str']}' với độ tin cậy: {pred['score']:.4f}")
		print(f"-> Câu hoàn chỉnh: {pred['sequence']}")

No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8 (https://huggingface.co/distilbert/distilroberta-base).
Using a pipeline without specifying a model name and revision in production is not recommended.
Some weights of the model checkpoint at distilbert/distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


Câu gốc: Hanoi is the <mask> of Vietnam.
Dự đoán: ' capital' với độ tin cậy: 0.9341
-> Câu hoàn chỉnh: Hanoi is the capital of Vietnam.
Dự đoán: ' Republic' với độ tin cậy: 0.0300
-> Câu hoàn chỉnh: Hanoi is the Republic of Vietnam.
Dự đoán: ' Capital' với độ tin cậy: 0.0105
-> Câu hoàn chỉnh: Hanoi is the Capital of Vietnam.
Dự đoán: ' birthplace' với độ tin cậy: 0.0054
-> Câu hoàn chỉnh: Hanoi is the birthplace of Vietnam.
Dự đoán: ' heart' với độ tin cậy: 0.0014
-> Câu hoàn chỉnh: Hanoi is the heart of Vietnam.


1. Mô hình đã dự đoán đúng từ capital không?
-> Mô hình dự đoán đúng từ capital
2. Tại sao các mô hình Encoder-only như BERT lại phù hợp cho tác vụ này?
-> Vì chúng dùng cơ chế attention hai chiều (bidirectional), cho phép mô hình nhìn cả ngữ cảnh trái và phải để dự đoán token bị che. Hơn nữa, MLM chính là mục tiêu pre-training gốc của BERT, nên kiến trúc này tối ưu cho tác vụ fill-mask và đạt độ chính xác cao.

# **Bài 2: Dự đoán từ tiếp theo (Next Token Prediction)**

In [10]:
from transformers import pipeline

# 1. Tải pipeline "text-generation"
# Pipeline này sẽ tự động tải một mô hình phù hợp (thường là GPT-2)
generator = pipeline("text-generation")

# 2. Đoạn văn bản mồi
prompt = "The best thing about learning NLP is"

# 3. Sinh văn bản
# max_length: tổng độ dài của câu mồi và phần được sinh ra
# num_return_sequences: số lượng chuỗi kết quả muốn nhận
generated_texts = generator(prompt, max_length=50, num_return_sequences=1)

# 4. In kết quả
print(f"Câu mồi: '{prompt}'")
for text in generated_texts:
	print("Văn bản được sinh ra:")
	print(text['generated_text'])

No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d (https://huggingface.co/openai-community/gpt2).
Using a pipeline without specifying a model name and revision in production is not recommended.
c:\Users\DELL\Downloads\NLP_DL\nlp-labs\nlp-labs\nlp_dl\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--openai-community--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see t

Câu mồi: 'The best thing about learning NLP is'
Văn bản được sinh ra:
The best thing about learning NLP is the flexibility and creativity that comes with it, it really makes you question why you can't do it without it, why you would never do it if you had the mental fortitude to do it yourself. It's amazing how much you can learn from people, especially when you're learning from people who are really experienced with NLP. It's a really valuable lesson, because it's not about your own mental fortitude, but rather about people's mental fortitude, and it shows how people can learn from each other.

This is a video from the podcast that you should watch if you're interested in learning NLP.

The new format of the podcast is called "NLP Talks," so it's probably not going to be popular. But it's already been a great way for people to learn NLP. There are some good lessons as well, and this podcast is definitely one of them.

What's your advice?

I hope that this podcast will help you get sta

1. Kết quả sinh ra có hợp lý không?
-> Có, kết quả hợp lý - ở mức ngữ pháp đúng, câu trôi chảy, và bám sát chủ đề.

2. Tại sao các mô hình Decoder-only như GPT lại phù hợp cho tác vụ này?
-> Vì chúng được huấn luyện theo mục tiêu dự đoán token kế tiếp (next-token prediction), sử dụng cơ chế causal attention một chiều. Điều này cho phép mô hình sinh văn bản theo trình tự, giữ mạch câu và chủ đề tốt hơn so với các mô hình Encoder-only như BERT, vốn được thiết kế cho nhiệm vụ hiểu và điền khuyết thay vì sinh văn bản.

# **Bài 3: Tính toán Vector biểu diễn của câu (Sentence Representation)**

In [13]:
import torch
from transformers import AutoTokenizer, AutoModel

# 1. Chọn một mô hình BERT
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# 2. Câu đầu vào
sentences = ["This is a sample sentence."]

# 3. Tokenize câu
# padding=True: đệm các câu ngắn hơn để có cùng độ dài
# truncation=True: cắt các câu dài hơn
# return_tensors='pt': trả về kết quả dưới dạng PyTorch tensors
inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')

# 4. Đưa qua mô hình để lấy hidden states
# torch.no_grad() để không tính toán gradient, tiết kiệm bộ nhớ
with torch.no_grad():
	outputs = model(**inputs)

# outputs.last_hidden_state chứa vector đầu ra của tất cả các token
last_hidden_state = outputs.last_hidden_state
# shape: (batch_size, sequence_length, hidden_size)

# 5. Thực hiện Mean Pooling
# Để tính trung bình chính xác, chúng ta cần bỏ qua các token đệm (padding tokens)
attention_mask = inputs['attention_mask']
mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
sum_embeddings = torch.sum(last_hidden_state * mask_expanded, 1)
sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
sentence_embedding = sum_embeddings / sum_mask

# 6. In kết quả
print("Vector biểu diễn của câu:")
print(sentence_embedding)
print("\nKích thước của vector:", sentence_embedding.shape)

Vector biểu diễn của câu:
tensor([[-6.3874e-02, -4.2837e-01, -6.6779e-02, -3.8430e-01, -6.5784e-02,
         -2.1826e-01,  4.7636e-01,  4.8659e-01,  4.0658e-05, -7.4274e-02,
         -7.4741e-02, -4.7635e-01, -1.9773e-01,  2.4824e-01, -1.2162e-01,
          1.6678e-01,  2.1045e-01, -1.4576e-01,  1.2636e-01,  1.8635e-02,
          2.4640e-01,  5.7090e-01, -4.7014e-01,  1.3782e-01,  7.3650e-01,
         -3.3808e-01, -5.0330e-02, -1.6452e-01, -4.3517e-01, -1.2900e-01,
          1.6516e-01,  3.4004e-01, -1.4930e-01,  2.2422e-02, -1.0488e-01,
         -5.1916e-01,  3.2964e-01, -2.2162e-01, -3.4206e-01,  1.1993e-01,
         -7.0148e-01, -2.3126e-01,  1.1224e-01,  1.2550e-01, -2.5191e-01,
         -4.6374e-01, -2.7261e-02, -2.8415e-01, -9.9250e-02, -3.7017e-02,
         -8.9192e-01,  2.5005e-01,  1.5816e-01,  2.2701e-01, -2.8497e-01,
          4.5300e-01,  5.0940e-03, -7.9441e-01, -3.1008e-01, -1.7403e-01,
          4.3029e-01,  1.6816e-01,  1.0590e-01, -4.8987e-01,  3.1856e-01,
          3.

1. Kích thước (chiều) của vector biểu diễn là bao nhiêu? Con số này tương ứng với tham số nào của mô hình BERT?
- Kích thước (chiều) của vector biểu diễn là 768.
- Con số này tương ứng với tham số hidden_size của mô hình BERT-base-uncased, tức là số chiều của vector ẩn mà mô hình sinh ra cho mỗi token trong lớp cuối của Transformer.
2. Tại sao chúng ta cần sử dụng attention_mask khi thực hiện Mean Pooling?
Nếu không loại bỏ padding, mean pooling sẽ tính trung bình luôn cả các vector [PAD] - những vector này không mang ý nghĩa ngữ nghĩa và có giá trị gần 0 => làm hỏng embedding.